In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data_postprocessing import obtain_shoreline
from src.data_processing.dataset_loader import CoastData
from scipy.spatial import cKDTree


In [3]:
def compute_distance(coords_pred, coords_gt):
    # Distance from predicted to GT
    # Create a KDTree for the Ground Truth coordinates
    tree_gt = cKDTree(coords_gt)

    # Find the nearest neighbors in the Ground Truth for each coordinate in the predicted mask
    dists_pred_to_gt, _ = tree_gt.query(coords_pred)

    # Distance from GT to predicted
    # Create a KDTree for the predicted coordinates
    tree_pred = cKDTree(coords_pred)
    # Find the nearest neighbors in the predicted for each coordinate in the GT mask
    dists_gt_to_pred, _ = tree_pred.query(coords_gt)

    return dists_pred_to_gt, dists_gt_to_pred

In [4]:
def calculate_dataset(data_path, in_pixels=True):
    data = CoastData(data_path)

    filtered_data = data.get_images(get_all_metadata=True, get_mask=False) # All the data

    global_distance_all_points = {
        "dist_pred_to_gt": [],
        "dist_gt_to_pred": [],
        "pred_points": 0,
        "gt_points": 0
    }

    for item in filtered_data:
        coords_gt_u = item['metadata']["image"]["shoreline"]["coordinates"]['u']
        coords_gt_v = item['metadata']["image"]["shoreline"]["coordinates"]['v']
        coords_gt = np.column_stack((coords_gt_u, coords_gt_v))

        coords_pred_u = item['metadata']['image']['predicted_shoreline']["coordinates"]['u']
        coords_pred_v = item['metadata']['image']['predicted_shoreline']["coordinates"]['v']
        coords_pred = np.column_stack((coords_pred_u, coords_pred_v))

        dists_pred_to_gt, dists_gt_to_pred = compute_distance(coords_pred, coords_gt)

        global_distance_all_points["dist_pred_to_gt"].extend(dists_pred_to_gt)
        global_distance_all_points["dist_gt_to_pred"].extend(dists_gt_to_pred)
        global_distance_all_points["pred_points"] += len(coords_pred)
        global_distance_all_points["gt_points"] += len(coords_gt)

    pixel_scale = 1 if in_pixels else 0.5

    global_distance_all_points["dist_pred_to_gt"] = np.array(global_distance_all_points["dist_pred_to_gt"]) * pixel_scale
    global_distance_all_points["dist_gt_to_pred"] = np.array(global_distance_all_points["dist_gt_to_pred"]) * pixel_scale

    mean_dist_pred_to_gt = np.mean(global_distance_all_points["dist_pred_to_gt"])
    std_dist_pred_to_gt = np.std(global_distance_all_points["dist_pred_to_gt"])
    rmsd_dist_pred_to_gt = np.sqrt(np.mean(np.square(global_distance_all_points["dist_pred_to_gt"])))
    mean_dist_gt_to_pred = np.mean(global_distance_all_points["dist_gt_to_pred"])
    std_dist_gt_to_pred = np.std(global_distance_all_points["dist_gt_to_pred"])
    rmsd_dist_gt_to_pred = np.sqrt(np.mean(np.square(global_distance_all_points["dist_gt_to_pred"])))
    q3_dist_pred_to_gt = np.percentile(global_distance_all_points["dist_pred_to_gt"], 75)
    q3_dist_gt_to_pred = np.percentile(global_distance_all_points["dist_gt_to_pred"], 75)

    ratio = global_distance_all_points["pred_points"] / global_distance_all_points["gt_points"]

    print(f"Number of points pred: {len(global_distance_all_points['dist_pred_to_gt'])}, Number of points GT: {len(global_distance_all_points['dist_gt_to_pred'])}, Ratio: {ratio:.4f}")
    print(f"Mean Absolute Distance (pred->Gt) global: {np.mean(mean_dist_pred_to_gt):.4f} ({std_dist_pred_to_gt:.4f})")
    print(f"Mean Absolute Distance (Gt->pred) global: {np.mean(mean_dist_gt_to_pred):.4f} ({std_dist_gt_to_pred:.4f})")
    print(f"RMSD (pred->Gt) global: {np.mean(rmsd_dist_pred_to_gt):.4f}")
    print(f"RMSD (Gt->pred) global: {np.mean(rmsd_dist_gt_to_pred):.4f}")
    print(f"Q3 75th Percentile (pred->Gt) global: {np.mean(q3_dist_pred_to_gt):.4f}")
    print(f"Q3 75th Percentile (Gt->pred) global: {np.mean(q3_dist_gt_to_pred):.4f}")



In [8]:
data_path_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/UNet/"))
data_path_attention_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/AttentionUNet/"))
data_path_deeplabv3 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/DeepLabV3/"))
data_path_ducknet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/DuckNet/"))

print("UNet results:")
calculate_dataset(data_path_unet, in_pixels=False)
print("\nAttention UNet results:")
calculate_dataset(data_path_attention_unet, in_pixels=False)
print("\nDeepLabV3 results:")
calculate_dataset(data_path_deeplabv3, in_pixels=False)
print("\nDuckNet results:")
calculate_dataset(data_path_ducknet, in_pixels=False)

UNet results:
CoastData: global - 174 images
Number of points pred: 102311, Number of points GT: 134174, Ratio: 0.7625
Mean Absolute Distance (pred->Gt) global: 6.0268 (9.2819)
Mean Absolute Distance (Gt->pred) global: 4.8741 (10.3773)
RMSD (pred->Gt) global: 11.0669
RMSD (Gt->pred) global: 11.4650
Q3 75th Percentile (pred->Gt) global: 6.8007
Q3 75th Percentile (Gt->pred) global: 5.0990

Attention UNet results:
CoastData: global - 174 images
Number of points pred: 103315, Number of points GT: 134174, Ratio: 0.7700
Mean Absolute Distance (pred->Gt) global: 6.5289 (10.9470)
Mean Absolute Distance (Gt->pred) global: 5.5863 (13.2066)
RMSD (pred->Gt) global: 12.7461
RMSD (Gt->pred) global: 14.3395
Q3 75th Percentile (pred->Gt) global: 6.7082
Q3 75th Percentile (Gt->pred) global: 5.0990

DeepLabV3 results:
CoastData: global - 174 images
Number of points pred: 97031, Number of points GT: 134174, Ratio: 0.7232
Mean Absolute Distance (pred->Gt) global: 4.9813 (6.9645)
Mean Absolute Distance (Gt

In [12]:
# Oblique results
data_path_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/UNet/"))
data_path_attention_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/AttentionUNet/"))
data_path_deeplabv3 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/DeepLabV3/"))
data_path_ducknet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/DuckNet/"))

print("UNet results:")
calculate_dataset(data_path_unet)
print("\nAttention UNet results:")
calculate_dataset(data_path_attention_unet)
print("\nDeepLabV3 results:")
calculate_dataset(data_path_deeplabv3)
print("\nDuckNet results:")
calculate_dataset(data_path_ducknet)

UNet results:
CoastData: global - 174 images
Number of points pred: 406665, Number of points GT: 341638, Ratio: 1.1903
Mean Absolute Distance (pred->Gt) global: 59.4567 (192.9484)
Mean Absolute Distance (Gt->pred) global: 62.7271 (239.6138)
RMSD (pred->Gt) global: 201.9015
RMSD (Gt->pred) global: 247.6883
Q3 75th Percentile (pred->Gt) global: 26.9258
Q3 75th Percentile (Gt->pred) global: 18.6815

Attention UNet results:
CoastData: global - 174 images
Number of points pred: 435346, Number of points GT: 341638, Ratio: 1.2743
Mean Absolute Distance (pred->Gt) global: 152.3757 (382.4166)
Mean Absolute Distance (Gt->pred) global: 119.6105 (385.6378)
RMSD (pred->Gt) global: 411.6562
RMSD (Gt->pred) global: 403.7613
Q3 75th Percentile (pred->Gt) global: 50.4480
Q3 75th Percentile (Gt->pred) global: 26.0000

DeepLabV3 results:
CoastData: global - 174 images
Number of points pred: 353246, Number of points GT: 341638, Ratio: 1.0340
Mean Absolute Distance (pred->Gt) global: 39.7567 (140.0585)
Mea